In [ ]:
!pip install fastapi uvicorn pyngrok nest-asyncio -q
print("Installation complete")

Installation complete


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import joblib

# Load the saved model and features
model = joblib.load('/content/drive/MyDrive/VoidWatch/model/voidwatch_model.pkl')
features = joblib.load('/content/drive/MyDrive/VoidWatch/model/features.pkl')

print("Model loaded successfully")
print(f"Expecting {len(features)} features")

Mounted at /content/drive
Model loaded successfully
Expecting 38 features


In [ ]:
!pip install pyngrok -q

from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
import pandas as pd
import numpy as np
import nest_asyncio
from pyngrok import ngrok
import uvicorn

nest_asyncio.apply()

# Initialize FastAPI app
app = FastAPI(title="VoidWatch API", version="1.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Define input structure
class ConjunctionInput(BaseModel):
    miss_distance: float
    mahalanobis_distance: float
    relative_position_r: float
    relative_position_t: float
    relative_position_n: float
    relative_speed: float
    time_to_tca: float
    c_sigma_r: float
    c_sigma_t: float
    c_sigma_n: float
    c_sigma_rdot: float
    c_sigma_tdot: float
    c_sigma_ndot: float
    t_sigma_r: float
    t_sigma_t: float
    t_sigma_n: float
    t_sigma_rdot: float
    t_sigma_tdot: float
    t_sigma_ndot: float
    t_j2k_inc: float
    t_j2k_sma: float
    t_j2k_ecc: float
    c_j2k_inc: float
    c_j2k_sma: float
    c_j2k_ecc: float
    t_h_apo: float
    t_h_per: float
    c_h_apo: float
    c_h_per: float
    t_obs_used: float
    c_obs_used: float
    t_weighted_rms: float
    c_weighted_rms: float
    relative_velocity_r: float
    relative_velocity_t: float
    relative_velocity_n: float
    F10: float
    AP: float

print("API structure defined")

API structure defined


In [ ]:
@app.get("/")
def home():
    return {"message": "VoidWatch API is live", "version": "1.0"}

@app.post("/predict")
def predict(data: ConjunctionInput):

    # Convert input to dataframe
    input_dict = data.dict()
    input_df = pd.DataFrame([input_dict])

    # Ensure correct feature order
    input_df = input_df[features]

    # Get prediction
    prediction = model.predict(input_df)[0]

    # Get probability scores for each class
    probabilities = model.predict_proba(input_df)[0]
    classes = model.classes_
    prob_dict = dict(zip(classes, probabilities.round(3).tolist()))

    # Get top risk factors
    feature_importances = pd.Series(
        model.feature_importances_,
        index=features
    ).sort_values(ascending=False)
    top_factors = feature_importances.head(3).index.tolist()
    top_factor_values = [
        f"{feat}: {round(input_dict[feat], 2)}"
        for feat in top_factors
    ]

    # Build recommendation
    time_to_tca = input_dict['time_to_tca']
    if prediction == 'HIGH':
        urgency = "Act immediately"
        maneuver_window = f"{round(time_to_tca * 24 * 0.5, 1)}-{round(time_to_tca * 24 * 0.7, 1)} hours from now"
    elif prediction == 'MEDIUM':
        urgency = "Monitor closely"
        maneuver_window = f"{round(time_to_tca * 24 * 0.6, 1)}-{round(time_to_tca * 24 * 0.8, 1)} hours from now"
    else:
        urgency = "No action required"
        maneuver_window = "N/A"

    return {
        "risk_level": str(prediction),
        "probabilities": {str(k): float(v) for k, v in zip(classes, probabilities)},
        "confidence": float(round(max(probabilities), 3)),
        "top_risk_factors": top_factor_values,
        "recommended_action": {
            "urgency": str(urgency),
            "maneuver_window": str(maneuver_window)
        }
    }

print("Endpoints defined")

Endpoints defined


In [ ]:
from pyngrok import ngrok
import nest_asyncio
import uvicorn
import asyncio

nest_asyncio.apply()

# Set token
ngrok.set_auth_token("3B7706Nrw2DTSmaCVpeFi7TzSeX_4ZqAAe6NiEL99EGFEPgxd")

# Kill any existing tunnels
ngrok.kill()

# Start fresh tunnel
public_url = ngrok.connect(8000)
print(f"VoidWatch API is live at: {public_url}")
print(f"Interactive docs at: {public_url}/docs")

# Run with nest_asyncio compatible config
config = uvicorn.Config(app, host="0.0.0.0", port=8000)
server = uvicorn.Server(config)
await server.serve()

VoidWatch API is live at: NgrokTunnel: "https://shemeka-goodish-overfrugally.ngrok-free.dev" -> "http://localhost:8000"
Interactive docs at: NgrokTunnel: "https://shemeka-goodish-overfrugally.ngrok-free.dev" -> "http://localhost:8000"/docs


INFO:     Started server process [6914]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


INFO:     103.186.199.208:0 - "OPTIONS /predict HTTP/1.1" 200 OK
INFO:     103.186.199.208:0 - "POST /predict HTTP/1.1" 200 OK


/tmp/ipykernel_6914/707437434.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_dict = data.dict()


INFO:     103.186.199.208:0 - "POST /predict HTTP/1.1" 200 OK


/tmp/ipykernel_6914/707437434.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_dict = data.dict()


INFO:     103.186.199.208:0 - "POST /predict HTTP/1.1" 200 OK


/tmp/ipykernel_6914/707437434.py:9: PydanticDeprecatedSince20: The `dict` method is deprecated; use `model_dump` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  input_dict = data.dict()
